In [1]:
import numpy as np 
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Two-Way ANOVA

#### Example
A physiologist was interested in learning whether smoking history and different types of stress tests influence the timing of a subject's maximum oxygen uptake, as measured in minutes. The researcher classified a subject's smoking history as either heavy smoking, moderate smoking, or non-smoking. He was interested in seeing the effects of three different types of stress tests — a test performed on a bicycle, a test on a treadmill, and a test on steps. The physiologist recruited 9 non-smokers, 9 moderate smokers, and 9 heavy smokers to participate in his experiment, for a total of $n = 27$ subjects. He then randomly assigned each of his recruited subjects to undergo one of the three types of stress tests.

Is there sufficient evidence at the $\alpha = 0.05$ significance level to conclude that smoking history has an effect on the time to maximum oxygen uptake? Is there sufficient evidence at the $\alpha = 0.05$ significance level to conclude that the type of stress test has an effect on the time to maximum oxygen uptake? And, is there evidence of an interaction between smoking history and the type of stress test? (Don't forget to define the null hypothesis $H_0$ and the alternative hypothesis $H_1$.)

## Hypotheses:
Main Effect of Smoking History
- $H_0: \mu_{\text{Non-Smoker}} = \mu_{\text{Moderate Smoker}} = \mu_{\text{Heavy Smoker}}$ (No difference in the timing of maximum oxygen uptake across smoking history groups)
- $H_1: \text{At least one group mean differs.}$ (At least one smoking history group differs in the timing of maximum oxygen uptake)

Main Effect of Stress Test Type
- $H_0: \mu_{\text{Bicycle}} = \mu_{\text{Treadmill}} = \mu_{\text{Step}}$ (No difference in the timing of maximum oxygen uptake across stress test types)
- $H_1: \text{At least one test mean differs.}$ (At least one stress test type differs in the timing of maximum oxygen uptake.)

Interaction Effect between Smoking History and Stress Test Type
- $H_0: \text{The effect of Smoking History on timing is the same across all stress test types.}$ (No interaction between smoking history and stress test type in affecting the timing of maximum oxygen uptake)
- $H_1: \text{The effect of Smoking History on timing differs by Stress Test type.} $ (The effect of smoking history on timing of maximum oxygen uptake depends on the type of stress test)

Independent Variables (Factors):
- Smoking Status (Nonsmoker, Moderate, Heavy)
- Exercise Test Type (Bicycle, Treadmill, Step Test)

Dependent Variable: 
- The numerical scores from the tests

In [2]:
# Here is the data smoking history vs test on bicyle, treadmill and step:
# Bicycle Test
bicycle_nonsmoker = [12.8, 13.5, 11.2]
bicycle_moderate = [10.9, 11.1, 9.8]
bicycle_heavy = [8.7, 9.2, 7.5]

# Treadmill Test
treadmill_nonsmoker = [16.2, 18.1, 17.8]
treadmill_moderate = [15.5, 13.8, 16.2]
treadmill_heavy = [14.7, 13.2, 8.1]

# Step Test
step_nonsmoker = [22.6, 19.3, 18.9]
step_moderate = [20.1, 21.0, 15.9]
step_heavy = [16.2, 16.1, 17.8]

### Reorganize data if necessary

In [3]:
data = {
    "Smoking": ["Nonsmoker"] * 3 + ["Moderate"] * 3 + ["Heavy"] * 3 +
               ["Nonsmoker"] * 3 + ["Moderate"] * 3 + ["Heavy"] * 3 +
               ["Nonsmoker"] * 3 + ["Moderate"] * 3 + ["Heavy"] * 3,
    "Test": ["Bicycle"] * 9 + ["Treadmill"] * 9 + ["Step"] * 9,
    "Score": [12.8, 13.5, 11.2, 10.9, 11.1, 9.8, 8.7, 9.2, 7.5,
              16.2, 18.1, 17.8, 15.5, 13.8, 16.2, 14.7, 13.2, 8.1,
              22.6, 19.3, 18.9, 20.1, 21.0, 15.9, 16.2, 16.1, 17.8]
}

# Convert to DataFrame
data = pd.DataFrame(data)
data

,Smoking,Test,Score
0,Nonsmoker,Bicycle,12.8
1,Nonsmoker,Bicycle,13.5
2,Nonsmoker,Bicycle,11.2
3,Moderate,Bicycle,10.9
4,Moderate,Bicycle,11.1
5,Moderate,Bicycle,9.8
6,Heavy,Bicycle,8.7
7,Heavy,Bicycle,9.2
8,Heavy,Bicycle,7.5
9,Nonsmoker,Treadmill,16.2


In [4]:
df = pd.DataFrame(data)

# Step 2: Calculate the overall mean
grand_mean = df['Score'].mean()

# Step 3: Calculate the sums of squares for each source of variation

# 3.1: Between groups (Smoking)
smoking_means = df.groupby('Smoking')['Score'].mean()
smoking_ss = sum(len(df[df['Smoking'] == level]) * (smoking_means[level] - grand_mean) ** 2 for level in smoking_means.index)

# 3.2: Between groups (Test)
test_means = df.groupby('Test')['Score'].mean()
test_ss = sum(len(df[df['Test'] == level]) * (test_means[level] - grand_mean) ** 2 for level in test_means.index)

# 3.3: Interaction (Smoking * Test)
interaction_ss = 0
for smoking_level in smoking_means.index:
    for test_level in test_means.index:
        subset = df[(df['Smoking'] == smoking_level) & (df['Test'] == test_level)]
        interaction_mean = subset['Score'].mean()
        interaction_ss += len(subset) * (interaction_mean - smoking_means[smoking_level] - test_means[test_level] + grand_mean) ** 2

# 3.4: Total sum of squares
total_ss = sum((df['Score'] - grand_mean) ** 2)

# 3.5: Error (Residual) sum of squares
error_ss = total_ss - smoking_ss - test_ss - interaction_ss

# Step 4: Calculate degrees of freedom (df)
df_smoking = len(smoking_means) - 1
df_test = len(test_means) - 1
df_interaction = df_smoking * df_test
df_error = len(df) - (df_smoking + df_test + df_interaction + 1)

# Step 5: Calculate Mean Squares (MS)
ms_smoking = smoking_ss / df_smoking
ms_test = test_ss / df_test
ms_interaction = interaction_ss / df_interaction
ms_error = error_ss / df_error

# Step 6: Calculate F-statistics
f_smoking = ms_smoking / ms_error
f_test = ms_test / ms_error
f_interaction = ms_interaction / ms_error

# Step 7: Calculate p-values
p_smoking = 1 - st.f.cdf(f_smoking, df_smoking, df_error)
p_test = 1 - st.f.cdf(f_test, df_test, df_error)
p_interaction = 1 - st.f.cdf(f_interaction, df_interaction, df_error)

# Step 8: Create an ANOVA table
anova_table = pd.DataFrame({
    'Source': ['Smoking', 'Test', 'Smoking * Test', 'Error'],
    'SS': [smoking_ss, test_ss, interaction_ss, error_ss],
    'df': [df_smoking, df_test, df_interaction, df_error],
    'MS': [ms_smoking, ms_test, ms_interaction, ms_error],
    'F': [f_smoking, f_test, f_interaction, np.nan],
    'p-value': [p_smoking, p_test, p_interaction, np.nan]
})

print(anova_table)


           Source          SS  df          MS          F       p-value
0         Smoking   84.898519   2   42.449259  12.896703  3.347912e-04
1            Test  298.071852   2  149.035926  45.279284  9.472739e-08
2  Smoking * Test    2.814815   4    0.703704   0.213795  9.273412e-01
3           Error   59.246667  18    3.291481        NaN           NaN


### Compare your from-scratch results with two-way ANOVA from the library statsmodels

In [5]:

# Fit two-way ANOVA model
model = ols("Score ~ Smoking*Test", data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                  sum_sq    df          F        PR(>F)
Smoking        84.898519   2.0  12.896703  3.347912e-04
Test          298.071852   2.0  45.279284  9.472739e-08
Smoking:Test    2.814815   4.0   0.213795  9.273412e-01
Residual       59.246667  18.0        NaN           NaN


In [6]:
print(p_smoking < 0.05)
print(p_test < 0.05)
print(p_interaction < 0.05)


True
True
False


### Please explain your results using $\alpha = 0.05$ significance level.

- Smoking: 

Since the p-value is much smaller than 0.05, we reject the null hypothesis. This means there is a significant main effect of Smoking on the scores.

- Test: 

The p-value is very small, so we reject the null hypothesis. There is a significant main effect of the Test type on the scores.

- Interaction:

Since the p-value is greater than 0.05, we fail to reject the null hypothesis. This indicates there is no significant interaction effect between Smoking and Test.
